# 🎬 Gudmarv Easy dubbing — on Colab

Fork: [5656wwed/SoniSlowVideo](https://github.com/5656wwed/SoniSlowVideo)

**What this does:** upload a video + SRT → crop/color/LUT → captions → pick a voice (Edge / Pocket / Kokoro) → optional BGM → dubs and renders. SRT is used as-is (no transcription needed). Optionally slow the final video to sync to the voice.

Voice cloning is done **from the website UI** (Pocket tab → upload a .wav → Create Clone).

### Run order
1. **Step 1** — install (run once, a few minutes)
2. **Step 1.5** — mount Drive so clones + outputs persist (optional but recommended)
3. **Step 2** — launch the web app and get a **public link**


In [ ]:
#@markdown ## Step 1: Install
import os, sys, subprocess

%cd /content
!rm -rf /content/SoniSlowVideo
!git clone -q https://github.com/5656wwed/SoniSlowVideo.git
%cd SoniSlowVideo
!git checkout -q dots-tts-clean 2>/dev/null || echo "dots-tts-clean not found, using main"

!pip uninstall chex pandas-stubs ibis-framework albumentations albucore jax numpy -y -q 2>/dev/null
!pip install -q uv==0.8.13
!uv venv --python 3.10 --clear -q
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!uv run python get-pip.py pip==23.1.2 -q
!uv run python -m pip install -q pip==23.1.2 Setuptools==80.6.0
!apt-get install -qq -y git-lfs ffmpeg 2>/dev/null
!git lfs install

# Install a static ffmpeg with NVENC so video encodes on the T4 GPU.
# (Stock Colab ffmpeg has NO hardware encoder.) Falls back to stock if it fails.
# Use the STABLE n7.1 ffmpeg build. The newest 'master' build needs NVENC API 13.1
# (driver 610+) which Colab's driver (API 13.0) does NOT support -> 'Driver does not
# support the required nvenc API version'. n7.1 uses an older NVENC API that works on
# Colab's T4 driver.
!mkdir -p /tmp/ffb && wget -qO /tmp/ffb.tar.xz https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-n7.1-latest-linux64-gpl-7.1.tar.xz
!tar -xf /tmp/ffb.tar.xz -C /tmp/ffb 2>/dev/null || true
!cp /tmp/ffb/ffmpeg-n7.1-latest-linux64-gpl-7.1/bin/ffmpeg /usr/local/bin/ffmpeg 2>/dev/null || true
!cp /tmp/ffb/ffmpeg-n7.1-latest-linux64-gpl-7.1/bin/ffprobe /usr/local/bin/ffprobe 2>/dev/null || true
# REAL test: encode a tiny clip with h264_nvenc. If it fails, NVENC won't work.
!ffmpeg -y -f lavfi -i testsrc=duration=0.2:size=64x64:rate=1 -c:v h264_nvenc -f null - 2>/tmp/nvenc_test.err && echo 'NVENC OK - GPU video encode works' || (echo 'NVENC BROKEN:'; tail -3 /tmp/nvenc_test.err)

# Detect GPU and install the right torch build.
import shutil
has_gpu = shutil.which('nvidia-smi') is not None
print('GPU detected:', has_gpu)
if has_gpu:
    subprocess.run('uv run python -m pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124', shell=True)
else:
    subprocess.run('uv run python -m pip install -q torch torchvision torchaudio', shell=True)

!sed -i 's|git+https://github.com/R3gm/whisperX.git@cuda_11_8|git+https://github.com/R3gm/whisperX.git@cuda_12_x|' requirements_base.txt
!uv run python -m pip install -q -r requirements_base.txt
!uv run python -m pip install -q -r requirements_extra.txt
!uv run python -m pip install -q onnxruntime-gpu==1.22.0 2>/dev/null || uv run python -m pip install -q onnxruntime
!uv run python -m pip install -q "gradio==4.19.2"   # engine imports gradio types

!uv run python -m pip install -q piper-tts==1.2.0 2>/dev/null
!uv run python -m pip install -q -r requirements_xtts.txt 2>/dev/null
!uv run python -m pip install -q TTS==0.21.1 --no-deps 2>/dev/null
!uv run python -m pip install -q kokoro
!uv run python -m pip install -q pocket-tts
!uv run python -m pip install -q edge-tts

!sudo apt-get install -y libcudnn8 -q 2>/dev/null || echo "libcudnn8 skipped"
!uv run python -m pip install -q "numpy<2.0" --force-reinstall --no-deps

# Install cloudflared BINARY to /usr/local/bin so the `cloudflared` command works.
!wget -qO /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!cloudflared --version 2>/dev/null || echo "cloudflared binary missing (network blocked?)"

print("\n✅ Installed. Run Step 2.\n")

In [ ]:
#@markdown ## Step 1.5: Persistent storage on Google Drive (clones + outputs)
# Cloned voices and rendered outputs are saved to your Drive so they survive
# Colab session restarts. Cloning is done from the WEBSITE UI, not here.
import os, shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

VOICE_DIR = '/content/drive/MyDrive/Gudmarv_Pocket_Voices'
os.makedirs(VOICE_DIR, exist_ok=True)

# The website saves new clones here -> they persist on Drive across sessions.
os.environ['POCKET_CLONE_DIR'] = VOICE_DIR

# Sync any existing clones/samples into the app's dropdown folder so they appear.
target = '/content/SoniSlowVideo/_POCKET_'
os.makedirs(target, exist_ok=True)
for fn in os.listdir(VOICE_DIR):
    shutil.copy(os.path.join(VOICE_DIR, fn), os.path.join(target, fn))

SAVE_DIR = "/content/drive/MyDrive/GudmarvEasyDubbing_Outputs"
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ['SAVE_DIR'] = SAVE_DIR

print('💾 Cloned voices ->', VOICE_DIR, '(from the website, persisted on Drive)')
print('💾 Rendered outputs ->', SAVE_DIR)
print('Continue to Step 2.')

In [ ]:
#@markdown ## Step 2: Launch the web app + public link
import os, re, subprocess, time, urllib.request

%cd /content/SoniSlowVideo

YOUR_HF_TOKEN = "" #@param {type:'string'}
os.environ['YOUR_HF_TOKEN']=YOUR_HF_TOKEN
os.environ['HF_TOKEN']=YOUR_HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN']=YOUR_HF_TOKEN

# Use GPU on Colab for faster Kokoro, else CPU.
try:
    import torch
    os.environ['SONI_CPU_MODE'] = '0' if torch.cuda.is_available() else '1'
except Exception:
    os.environ['SONI_CPU_MODE'] = '1'
print('SONI_CPU_MODE =', os.environ['SONI_CPU_MODE'], '(0=GPU, 1=CPU)')
os.environ['SONI_LOG_FILE'] = '/content/server.log'   # where web_app.py stdout goes on Colab

log = open('/content/server.log','w')
server = subprocess.Popen(['uv','run','python','web_app.py'], stdout=log, stderr=subprocess.STDOUT)

up = False
for _ in range(150):
    try:
        urllib.request.urlopen('http://127.0.0.1:7860/', timeout=2); up=True; break
    except Exception:
        time.sleep(2)
print('✅ web app up on http://127.0.0.1:7860' if up else '❌ not up — see /content/server.log')
if not up:
    print(open('/content/server.log').read()[-1500:])

# Free public tunnel (cloudflared quick tunnel) — binary installed in Step 1.
tlog = open('/content/tunnel.log','w')
tunnel = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],
                          stdout=tlog, stderr=subprocess.STDOUT)
time.sleep(12)
url = None
try:
    for line in open('/content/tunnel.log'):
        m = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
        if m: url = m.group(1); break
except Exception:
    pass
if url:
    print('\n🌐 OPEN THIS IN YOUR BROWSER:\n')
    print(url)
    from IPython.display import display, HTML
    display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))
else:
    print('Tunnel log so far:'); print(open('/content/tunnel.log').read()[-800:])